

# %% Step 2: Repeat the honest test 5 times and take the average
# Each repeat puts the 428 mixes into 5 groups at random. The model is trained on 4 groups and
# tested on the 5th, so it is always tested on mixes it has never seen.
# This takes about 2 minutes.
mixes = data["mix_id"].unique()
scores = []

for repeat in range(5):
    shuffled = np.random.RandomState(repeat).permutation(mixes)
    group_of_mix = {mix: position % 5 for position, mix in enumerate(shuffled)}
    group = data["mix_id"].map(group_of_mix)

    predictions = np.zeros(len(data))
    for g in range(5):
        train_rows = group != g
        check_rows = group == g
        repeat_model = GradientBoostingRegressor(
            n_estimators=800, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=0)
        repeat_model.fit(data.loc[train_rows, columns], y[train_rows])
        predictions[check_rows] = repeat_model.predict(data.loc[check_rows, columns])

    mae = mean_absolute_error(y, predictions)
    r2 = r2_score(y, predictions)
    scores.append((mae, r2))
    print("Repeat", repeat + 1, "-> MAE", round(mae, 2), "MPa | R2", round(r2, 3))

maes = [s[0] for s in scores]
r2s = [s[1] for s in scores]
print(f"\nAverage MAE {np.mean(maes):.2f} MPa (range {min(maes):.2f} - {max(maes):.2f}) | "
      f"average R2 {np.mean(r2s):.3f}")

# %% Step 3: A helper that predicts strength for any mix
# Give it the 7 ingredients (kg per m3) and the age (days). It works out the engineer's columns
# the same way as Day 2, then asks the model.
def predict_strength(cement, slag, fly_ash, water, superplasticizer, coarse_agg, fine_agg, age):
    binder = cement + slag + fly_ash
    row = pd.DataFrame([{
        "cement": cement, "slag": slag, "fly_ash": fly_ash, "water": water,
        "superplasticizer": superplasticizer, "coarse_agg": coarse_agg, "fine_agg": fine_agg,
        "age": age, "log_age": np.log(age), "binder": binder,
        "w_c": water / cement, "w_b": water / binder,
        "slag_share": slag / binder, "fly_ash_share": fly_ash / binder,
        "sp_per_binder": superplasticizer / binder,
        "fine_share": fine_agg / (fine_agg + coarse_agg),
    }])
    return model.predict(row[columns])[0]


# Try it on a typical mix: 350 kg cement, 175 kg water (w/c 0.50), 28 days
print("Example, 28 days:", round(predict_strength(350, 0, 0, 175, 5, 980, 780, 28), 1), "MPa")

# %% Step 4: Effect of the water/binder ratio (28 days)
# Keep the water at 180 kg and change the cement content to get each ratio
# (this is how a lower ratio is usually achieved: more cement for the same water).
# We stay between w/b 0.35 and 0.65, where the data has plenty of examples.
ratios = np.arange(0.35, 0.66, 0.05)
strengths = [predict_strength(180 / r, 0, 0, 180, 5, 980, 780, 28) for r in ratios]

for r, s in zip(ratios, strengths):
    print(f"w/b {r:.2f} -> {s:.1f} MPa")

plt.figure(figsize=(7, 4))
plt.plot(ratios, strengths, marker="o")
plt.xlabel("Water / binder ratio")
plt.ylabel("Predicted 28-day strength (MPa)")
plt.title("Lower water/cement ratio, higher strength (180 kg water, cement only)")
plt.tight_layout()
plt.savefig("effect_of_wb.png", dpi=150)
plt.show()

# %% Step 5: Strength gain over time for three kinds of binder
# Same total binder (400 kg) and the same water (180 kg, w/b 0.45) in all three mixes.
# Only the TYPE of binder changes.
ages = [3, 7, 14, 28, 56, 90, 180]
mixes_to_compare = {
    "Cement only (400 cement)": (400, 0, 0),
    "Slag blend (280 cement + 120 slag)": (280, 120, 0),
    "Fly ash blend (300 cement + 100 fly ash)": (300, 0, 100),
}

plt.figure(figsize=(8, 4.5))
for name, (cement, slag, fly_ash) in mixes_to_compare.items():
    curve = [predict_strength(cement, slag, fly_ash, 180, 5, 980, 780, age) for age in ages]
    plt.plot(ages, curve, marker="o", label=name)
    print(name)
    print("   7 days:", round(curve[1], 1), "| 28 days:", round(curve[3], 1), "| 90 days:", round(curve[5], 1),
          "| 7-day as share of 28-day:", round(curve[1] / curve[3], 2))

plt.xscale("log")
plt.xticks(ages, ages)
plt.xlabel("Age (days)")
plt.ylabel("Predicted strength (MPa)")
plt.title("Predicted strength gain: same binder content, different binder type")
plt.legend()
plt.tight_layout()
plt.savefig("strength_gain_by_binder.png", dpi=150)
plt.show()

# %% Step 6: Check against the REAL lab results (not the model)
# For mixes crushed at both 7 and 28 days, compare the 7-day share by type of binder.
# (There are no fly-ash-only mixes with both 7 and 28-day results, so that group may not appear.)
mix_info = data.groupby("mix_id")[["slag", "fly_ash"]].first()
by_age = data.groupby(["mix_id", "age"])["strength"].mean().unstack("age")
both = by_age[[7, 28]].dropna()
both = both.join(mix_info)


def binder_type(row):
    if row["slag"] == 0 and row["fly_ash"] == 0:
        return "Cement only"
    if row["slag"] > 0 and row["fly_ash"] == 0:
        return "With slag"
    if row["slag"] == 0 and row["fly_ash"] > 0:
        return "With fly ash"
    return "Slag and fly ash"


both["binder_type"] = both.apply(binder_type, axis=1)
both["share_7_of_28"] = both[7] / both[28]
print(both.groupby("binder_type")["share_7_of_28"].agg(["count", "median"]).round(2))

# %% Step 7: Which columns does the model lean on most?
# Note: columns that carry the same information (like w/b and binder) share the credit,
# so read this as a rough guide, not an exact ranking.
importance = pd.Series(model.feature_importances_, index=columns).sort_values()
plt.figure(figsize=(7, 5))
importance.plot(kind="barh")
plt.xlabel("Share of the model's decisions")
plt.title("What the model uses most")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()
print(importance.sort_values(ascending=False).round(3).head(8))

# %% Step 8: Your Day 4 notes (for the README)
#   "Honest error, averaged over 5 repeats: MAE about 3.5 MPa, R2 about 0.90."
#   "Each 0.05 increase in w/b costs roughly X MPa at 28 days (fill in from Step 4)."
#   "Slag and fly ash blends gain strength more slowly: 7-day share of 28-day is lower (Steps 5 and 6)."
#   "Age and the water/binder ratio are the main drivers (Step 7)."

In [2]:
# Concrete Compressive Strength - Day 4 (a stable score and practical answers)
# Goal 1: measure the honest error several times and report the average (Day 3 showed it moves
#         between about 3.3 and 3.8 MPa depending on how the mixes are split).
# Goal 2: use the model to answer questions a materials technician asks:
#         - how much strength do we lose as the water/binder ratio goes up?
#         - how do slag and fly ash change early and late strength?
# Run one cell at a time, in the folder with clean_concrete.csv and concrete_model.joblib.

# %% Step 1: Load the data and the saved model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

data = pd.read_csv("clean_concrete.csv")
data["log_age"] = np.log(data["age"])
columns = ["cement", "slag", "fly_ash", "water", "superplasticizer", "coarse_agg", "fine_agg", "age",
           "log_age", "binder", "w_c", "w_b", "slag_share", "fly_ash_share", "sp_per_binder", "fine_share"]
y = data["strength"]

model = joblib.load("concrete_model.joblib")       # the model trained on all the data on Day 3
print("Model loaded")

FileNotFoundError: [Errno 2] No such file or directory: 'clean_concrete.csv'